# 02 — PySINDy + EMLLibrary Demo

This notebook demonstrates how to use EML observables as a PySINDy feature library.

**Prerequisites:** `pip install pysindy`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import sys
sys.path.insert(0, '..')

## 1. Generate Lorenz Data

In [ ]:
def lorenz(t, state, sigma=10, rho=28, beta=8/3):
    x, y, z = state
    return [sigma*(y-x), x*(rho-z)-y, x*y - beta*z]

dt = 0.001
t_span = (0, 10)
t_eval = np.arange(0, 10, dt)
sol = solve_ivp(lorenz, t_span, [1, 1, 25], t_eval=t_eval, rtol=1e-10)
X = sol.y.T

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(X[:, 0], X[:, 1], X[:, 2], lw=0.5)
ax.set_title('Lorenz Attractor')
plt.show()
print(f'Shape: {X.shape}')

## 2. EMLLibrary Feature Generation

The `EMLLibrary` enumerates all EML tree compositions to a given depth, producing a feature matrix.

In [ ]:
from koopman_eml.sindy import EMLLibrary, eml_enumerate

# See what expressions depth=1 produces
exprs = eml_enumerate(1, n_vars=3, include_identity=False)
print(f'{len(exprs)} EML expressions at depth <= 1:')
for name, _ in exprs[:15]:
    print(f'  {name}')
print(f'  ... ({len(exprs) - 15} more)' if len(exprs) > 15 else '')

## 3. SINDy with EML Features

In [ ]:
try:
    import pysindy as ps

    eml_lib = EMLLibrary(depth=1, n_vars=3)
    model = ps.SINDy(feature_library=eml_lib, optimizer=ps.STLSQ(threshold=0.5))
    model.fit(X, t=dt)
    model.print()

    print(f'\nFeature library size: {eml_lib.n_output_features_}')
    print(f'Coefficients shape: {model.coefficients().shape}')
except ImportError:
    print('PySINDy not installed. Run: pip install pysindy')

## 4. Comparison: EML vs Polynomial Library

Compare SINDy with EML features against the standard polynomial library.

In [ ]:
try:
    import pysindy as ps

    # Polynomial baseline
    poly_model = ps.SINDy(feature_library=ps.PolynomialLibrary(degree=3))
    poly_model.fit(X, t=dt)

    print('=== Polynomial Library ===')
    poly_model.print()
    print(f'Number of features: {poly_model.n_features_in_}')
    print(f'Number of terms: {np.count_nonzero(poly_model.coefficients())}')

    print('\n=== EML Library ===')
    eml_model = ps.SINDy(feature_library=EMLLibrary(depth=1, n_vars=3),
                         optimizer=ps.STLSQ(threshold=0.5))
    eml_model.fit(X, t=dt)
    eml_model.print()
    print(f'Number of features: {eml_model.n_features_in_}')
    print(f'Number of terms: {np.count_nonzero(eml_model.coefficients())}')
except ImportError:
    print('PySINDy not installed.')